# 5) Using the model: sensitivity, scenarios and a policy brief

## Objectives

At the end of this notebook you should be able to:

- turn a dispatch model into an **analysis tool**: a function from assumptions to a small set of results
- separate **resource cost** from **carbon payments**, and **system cost** from what **consumers pay**
- run a **one-at-a-time sensitivity analysis** and show it as a tornado chart
- compare **policy scenarios**, compute **abatement costs** (€/t CO₂) and find **interactions** between measures
- write a **one-page policy brief** whose numbers come straight from the model

Lessons 2–4 built models and checked them. This lesson asks the questions a ministry, a regulator or a utility would ask: what if the CO₂ price doubles, what if Croatia leaves coal, is solar plus storage worth more than either alone? The answers come from a full **synthetic, Croatia-sized year** (M2.2) with lesson 3's ramp limits and battery. All costs and fuel prices are **illustrative**.

In [1]:
import logging
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

### The model as a function

M2.2's synthetic year (same seed, same profiles), now with lesson 3's operational detail: coal ramps at most 60 MW/h, the CCGT 300 MW/h, and a 300 MW battery. Every assumption we want to vary is an argument of `build`:

In [2]:
rng = np.random.default_rng(7)
idx = pd.date_range("2025-01-01", periods=8760, freq="h")
hour, doy = idx.hour.to_numpy(), idx.dayofyear.to_numpy()
load = (2050 + 350 * np.cos(2 * np.pi * (doy - 15) / 365) + 300 * np.sin(np.pi * (hour - 6) / 12).clip(0)
        - 150 * (idx.dayofweek.to_numpy() >= 5) + rng.normal(0, 60, 8760))
w = np.zeros(8760)
shocks = rng.normal(0, 1, 8760)
for t in range(1, 8760):
    w[t] = 0.97 * w[t - 1] + 0.24 * shocks[t]
wind_cf = np.clip(0.27 + 0.18 * w + 0.06 * np.cos(2 * np.pi * (doy - 15) / 365), 0, 0.95)
sun = np.clip(np.sin(np.pi * (hour - 6) / 13), 0, None) * (0.65 + 0.35 * np.sin(2 * np.pi * (doy - 80) / 365))
cloud = np.clip(1 - 0.5 * rng.beta(2, 3, 365).repeat(24), 0.2, 1)
solar_cf = np.clip(sun * cloud * 0.95, 0, 1)
ror_cf = np.clip(0.55 + 0.25 * np.sin(2 * np.pi * (doy - 60) / 365), 0.2, 0.9)

EMISSIONS = {"coal": 0.34 / 0.38, "ccgt": 0.20 / 0.55, "imports": 0.20 / 0.55, "ocgt": 0.20 / 0.38}   # t CO2 / MWh_el
BASE = {"solar": 1000, "coal": 600, "battery_mwh": 1200, "co2": 70.0, "gas": 35.0}


def build(solar=1000, coal=600, battery_mwh=1200, co2=70.0, gas=35.0):
    """The synthetic Croatian year. Capacities in MW / MWh, CO2 price in EUR/t, gas price in EUR/MWh_th."""
    mc = lambda fuel, eff, ef: (fuel + co2 * ef) / eff
    n = pypsa.Network()
    n.set_snapshots(idx)
    n.add("Bus", "HR")
    n.add("Load", "load", bus="HR", p_set=load)

    def gen(name, p_nom, p_max_pu, cost, ramp=None):
        if p_nom > 0:
            limits = {} if ramp is None else {"ramp_limit_up": ramp / p_nom, "ramp_limit_down": ramp / p_nom}
            n.add("Generator", name, bus="HR", p_nom=p_nom, p_max_pu=p_max_pu, marginal_cost=cost, **limits)

    gen("solar", solar, solar_cf, 0.0)
    gen("wind", 1000, wind_cf, 0.0)
    gen("run_of_river", 500, ror_cf, 0.0)
    gen("nuclear_share", 350, 1.0, 12.0)
    gen("coal", coal, 1.0, mc(12, 0.38, 0.34), ramp=60)
    gen("ccgt", 900, 1.0, mc(gas, 0.55, 0.20), ramp=300)
    gen("imports", 800, 1.0, mc(gas, 0.55, 0.20) + 15)
    gen("ocgt", 400, 1.0, mc(gas, 0.38, 0.20))
    if battery_mwh > 0:
        n.add("StorageUnit", "battery", bus="HR", p_nom=300, max_hours=battery_mwh / 300,
              efficiency_store=0.95, efficiency_dispatch=0.95, cyclic_state_of_charge=True)
    return n


def evaluate(**assumptions):
    """Solve one year and return the results a decision-maker needs (money in MEUR/year)."""
    a = {**BASE, **assumptions}
    n = build(**a)
    n.optimize(solver_name="highs", log_to_console=False, progress=False)
    price, gen = n.buses_t.marginal_price["HR"], n.generators_t.p
    emissions = sum((gen[g] * f).sum() for g, f in EMISSIONS.items() if g in gen)
    battery = n.storage_units_t.p["battery"] if a["battery_mwh"] > 0 else 0.0
    # every MWh consumed is paid at the hourly price - check that it equals what producers and the battery receive
    assert np.isclose((price * load).sum(), (price * (gen.sum(axis=1) + battery)).sum(), rtol=1e-6)
    res = [g for g in ["solar", "wind"] if g in gen]
    available = sum((n.generators.p_nom[g] * n.generators_t.p_max_pu[g]).sum() for g in res)
    return {"system_cost": n.objective / 1e6,
            "carbon_payments": a["co2"] * emissions / 1e6,
            "resource_cost": n.objective / 1e6 - a["co2"] * emissions / 1e6,
            "consumer_cost": (price * load).sum() / 1e6,
            "avg_price": float(price.mean()),
            "emissions_Mt": emissions / 1e6,
            "curtailment_GWh": (available - gen[res].sum().sum()) / 1e3,
            "res_share": float(gen[res + ["run_of_river"]].sum().sum() / load.sum())}


base = evaluate()
pd.Series(base).round(3)

system_cost         845.789
carbon_payments     298.106
resource_cost       547.683
consumer_cost      1727.126
avg_price            93.434
emissions_Mt          4.259
curtailment_GWh       0.000
res_share             0.348
dtype: float64

Four numbers describe the same year, and a policy discussion needs all four:

- **System cost** (845.8 MEUR) is the model's objective: fuel, operating costs *and* CO₂ payments at 70 €/t.
- **Carbon payments** (298.1 MEUR) are part of it, but they are a **transfer** — the money goes to the state budget through the EU ETS, it does not disappear. The **resource cost** (547.7 MEUR) is what the year really uses up in fuel and operation.
- **Consumer cost** (1,727 MEUR) is what consumers pay at the hourly prices — about twice the system cost, because under uniform pricing every MWh is paid the marginal price (M2.1's inframarginal rents).
- **Emissions** are 4.26 Mt CO₂; wind, solar and hydro cover 35 % of demand, with no curtailment.

The assertion inside `evaluate` checks an accounting identity in every run: what consumers pay equals what producers and the battery receive. If it ever failed, the model or the bookkeeping would be wrong.

### One-at-a-time sensitivity: what matters most?

Vary each uncertain input between a low and a high value, keeping everything else at the base case. The ranges are deliberately wide:

In [3]:
ranges = {"co2": (40.0, 100.0), "gas": (25.0, 45.0), "solar": (500, 2000), "battery_mwh": (0, 2400)}
labels = {"co2": "CO2 price 40 / 100 EUR/t", "gas": "gas price 25 / 45 EUR/MWh", "solar": "solar 500 / 2,000 MW",
          "battery_mwh": "battery 0 / 2,400 MWh"}
sensitivity = {}
for param, (low, high) in ranges.items():
    sensitivity[(param, "low")] = evaluate(**{param: low})
    sensitivity[(param, "high")] = evaluate(**{param: high})
sens = pd.DataFrame(sensitivity).T
sens.index.names = ["parameter", "case"]
sens[["system_cost", "resource_cost", "consumer_cost", "emissions_Mt", "avg_price"]].round(2)

                  system_cost  resource_cost  consumer_cost  emissions_Mt  \
parameter   case                                                            
co2         low        683.94         446.13        1447.12          5.95   
            high       961.45         634.81        2026.72          3.27   
gas         low        701.03         472.58        1492.14          3.26   
            high       933.13         516.96        1976.84          5.95   
solar       low        914.18         586.52        1741.93          4.68   
            high       721.66         471.57        1618.50          3.57   
battery_mwh low        846.26         546.20        1723.72          4.29   
            high       845.75         547.49        1727.34          4.26   

                  avg_price  
parameter   case             
co2         low       78.14  
            high     109.45  
gas         low       80.38  
            high     106.87  
solar       low       94.18  
            high   

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, metric, unit, title in [(axes[0], "system_cost", "MEUR/year", "system cost"),
                                (axes[1], "emissions_Mt", "Mt CO2/year", "emissions")]:
    delta = pd.DataFrame({case: sens.xs(case, level="case")[metric] - base[metric] for case in ["low", "high"]})
    delta = delta.loc[delta.abs().max(axis=1).sort_values().index]          # largest swing on top
    y = np.arange(len(delta))
    ax.barh(y, delta["low"], color="tab:blue", label="low value")
    ax.barh(y, delta["high"], color="tab:orange", label="high value")
    ax.set_yticks(y, [labels[p] for p in delta.index])
    ax.axvline(0, color="black", lw=0.8)
    ax.set(xlabel=f"change from base ({unit})", title=title)
axes[0].legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

The CO₂ price and the gas price dominate the system cost: moving them across their ranges shifts the cost by −162 to +116 MEUR and −145 to +87 MEUR per year. Solar capacity comes next (+68 / −124 MEUR). The battery barely registers (+0.47 / −0.04 MEUR) — a sharp contrast to lesson 3's sunny day, where the same battery saved 118,000 € in a single day. In this year nothing is curtailed and the gap between coal and gas costs is small, so there is little cheap energy to shift. **Storage is only as valuable as the system around it makes it.**

The emissions panel shows something else: the low CO₂ price and the high gas price give *identical* emissions (5.95 Mt), and the high CO₂ price and the low gas price give almost the same (3.27 and 3.26 Mt — the ramp limits and the battery behave slightly differently). Emissions do not respond smoothly — they jump when the **order of the merit order changes** (M2.2). At 40 €/t, or with gas at 45 €/MWh, coal becomes cheaper than gas; at 100 €/t, or with gas at 25 €/MWh, coal falls behind even the imports. Any input that causes the same switch gives the same dispatch. Notice also how **asymmetric** most bars are: the model is not linear in its inputs, so a symmetric range around the base does not give a symmetric answer.

### Policy scenarios

Sensitivities vary what we are *unsure* about. Scenarios combine what a government could *decide*. Four options, each against the base case:

In [5]:
scenarios = {
    "coal exit": {"coal": 0},
    "+1 GW solar": {"solar": 2000},
    "+1 GW solar + battery x2": {"solar": 2000, "battery_mwh": 2400},
    "coal exit + solar + battery": {"coal": 0, "solar": 2000, "battery_mwh": 2400},
}
results = pd.DataFrame({"base": base, **{name: evaluate(**s) for name, s in scenarios.items()}}).T
change = results - results.loc["base"]
abatement = (change["resource_cost"] / -change["emissions_Mt"]).where(change["emissions_Mt"] < -1e-9)
summary = pd.DataFrame({"system_cost_change": change["system_cost"], "resource_cost_change": change["resource_cost"],
                        "consumer_cost_change": change["consumer_cost"],
                        "emissions_change_Mt": change["emissions_Mt"], "abatement_cost_EUR_per_t": abatement,
                        "avg_price": results["avg_price"], "res_share": results["res_share"]})
summary.round(2)

                             system_cost_change  resource_cost_change  \
base                                       0.00                  0.00   
coal exit                                 18.14                 88.26   
+1 GW solar                             -124.13                -76.11   
+1 GW solar + battery x2                -127.61                -78.20   
coal exit + solar + battery             -113.94                 -8.53   

                             consumer_cost_change  emissions_change_Mt  \
base                                         0.00                 0.00   
coal exit                                  106.15                -1.00   
+1 GW solar                               -108.62                -0.69   
+1 GW solar + battery x2                   -79.91                -0.71   
coal exit + solar + battery                  3.29                -1.51   

                             abatement_cost_EUR_per_t  avg_price  res_share  
base                                  

Read the coal exit row carefully. The **system cost** rises by only 18 MEUR — but that figure already counts the carbon payments that are no longer made (about 70 €/t × 1.0 Mt = 70 MEUR). The **resource cost** rises by 88 MEUR, so cutting 1.00 Mt costs **88 €/t** in real resources: the honest abatement cost. Consumers pay 106 MEUR more, because the average price rises from 93.4 to 98.9 €/MWh on every MWh they buy.

The solar scenarios show *negative* abatement costs (about −111 €/t): emissions and running costs both fall. That is not a free lunch — the model includes **no investment cost** for the new solar or storage, so these are operating savings that the investment must pay for. Judging whether it does is a capacity-expansion question (Module 05).

Adding the larger battery to the solar scenario lowers the running cost a little further (−78.2 instead of −76.1 MEUR) but makes **consumers pay 29 MEUR more** (−79.9 instead of −108.6 MEUR): the battery buys solar in the cheapest hours and so lifts those prices. What is good for the system is not automatically good for every consumer bill. The full package cuts emissions by 1.51 Mt, lowers the running cost by 8.5 MEUR and leaves the average price almost where it was.

In [6]:
# does the battery add more value when there is more solar? (an interaction effect)
battery_alone = results.loc["base", "resource_cost"] - sens.loc[("battery_mwh", "high"), "resource_cost"]
battery_with_solar = results.loc["+1 GW solar", "resource_cost"] - results.loc["+1 GW solar + battery x2", "resource_cost"]
print(f"saving from doubling the battery: {battery_alone:.2f} MEUR/year in the base system, "
      f"{battery_with_solar:.2f} MEUR/year with 1 GW more solar")

saving from doubling the battery: 0.20 MEUR/year in the base system, 2.09 MEUR/year with 1 GW more solar


In the base system, a larger battery is almost worthless; with 1 GW more solar, the same extra battery saves about ten times as much. Solar creates the cheap midday energy that storage needs. A one-at-a-time sensitivity could never reveal this — it changes one input at a time by construction. **Interactions** like this are why policy packages are evaluated as packages, and why it is worth running a few combined scenarios even when each measure was studied on its own.

### A one-page policy brief

Decision-makers read one page. A good brief states the question, the answer in one sentence, three or four numbers that support it, what drives the result, and what the model cannot tell you. Writing it **from the results in code** guarantees that every number in the text is the model's number — rerun the model with new assumptions and the brief updates itself:

In [7]:
coal = summary.loc["coal exit"]
combo = summary.loc["coal exit + solar + battery"]
gas_swing = sens.loc[("gas", "high"), "system_cost"] - sens.loc[("gas", "low"), "system_cost"]
brief = f"""
POLICY BRIEF - Leaving coal in a Croatia-sized power system (synthetic data, illustrative costs)

Question: what does a coal exit cost, and what does it achieve?

Answer: on its own, a coal exit cuts emissions by {-coal.emissions_change_Mt:.2f} Mt CO2 per year at a resource
cost of {coal.abatement_cost_EUR_per_t:.0f} EUR/t; combined with 1 GW of extra solar and a larger battery, emissions fall by
{-combo.emissions_change_Mt:.2f} Mt while the running cost of the system (before any investment) falls by
{-combo.resource_cost_change:.0f} MEUR per year.

Key numbers (change against today's system, per year):
- Coal exit alone: running cost {coal.resource_cost_change:+.0f} MEUR, consumers pay {coal.consumer_cost_change:+.0f} MEUR,
  average price {coal.avg_price:.1f} EUR/MWh (today {summary.loc['base', 'avg_price']:.1f}).
- Coal exit + solar + battery: running cost {combo.resource_cost_change:+.0f} MEUR, consumers pay {combo.consumer_cost_change:+.0f} MEUR,
  renewable share {combo.res_share:.0%} (today {summary.loc['base', 'res_share']:.0%}).

What drives the result: fuel and carbon prices dominate - a gas price between 25 and 45 EUR/MWh alone moves
the system cost by {gas_swing:.0f} MEUR per year, more than any of the policies above.

What the model cannot tell you: the investment cost of the new solar and storage, which the running-cost saving
must pay for (Module 05), network limits (Module 04),
security of supply in a dry or windless year, and the cost of the coal plant's closure.
"""
print(brief)


POLICY BRIEF - Leaving coal in a Croatia-sized power system (synthetic data, illustrative costs)

Question: what does a coal exit cost, and what does it achieve?

Answer: on its own, a coal exit cuts emissions by 1.00 Mt CO2 per year at a resource
cost of 88 EUR/t; combined with 1 GW of extra solar and a larger battery, emissions fall by
1.51 Mt while the running cost of the system (before any investment) falls by
9 MEUR per year.

Key numbers (change against today's system, per year):
- Coal exit alone: running cost +88 MEUR, consumers pay +106 MEUR,
  average price 98.9 EUR/MWh (today 93.4).
- Coal exit + solar + battery: running cost -9 MEUR, consumers pay +3 MEUR,
  renewable share 43% (today 35%).

What drives the result: fuel and carbon prices dominate - a gas price between 25 and 45 EUR/MWh alone moves
the system cost by 232 MEUR per year, more than any of the policies above.

What the model cannot tell you: the investment cost of the new solar and storage, which the running-co

Notice how the brief is built. The answer comes first, in one sentence. Every number is a model output, formatted rather than typed. It separates the running cost of the system from what consumers pay, names the inputs that drive the result, and says plainly what the model leaves out — here, above all, the investment cost that the running-cost savings must cover. If the minister asks "what if gas is cheaper?", you change one argument and rerun, and the brief rewrites itself.

## Check your understanding

**Reading the results**

1. Why do consumers' payments change so much more than the resource cost when coal leaves the system? Who receives the difference?
2. A colleague reports "a coal exit costs only 18 MEUR a year". What did they compare, and why is it misleading?

**Method**

1. One-at-a-time sensitivities miss interactions. Give an example from this lesson, and say how you would find others.
2. The brief says fuel prices matter more than the policies. Does that mean the policies are unimportant? How would you present this to a minister?